# 07. Итоговое практическое задание

## Кейс: «РегионМаркет»

В этом ноутбуке нужно самостоятельно пройти полный мини-пайплайн аналитика:

```text
загрузка нескольких файлов
↓
проверка качества данных
↓
минимальная очистка
↓
объединение таблиц
↓
создание revenue
↓
расчет показателей
↓
построение графиков
↓
сохранение результата
↓
2–3 аналитических вывода
```

Это итоговое задание по теме **«Загрузка и интеграция данных из различных форматов. Инструменты для сбора данных. Основы Python для обработки данных»**.

## Что нужно сдать

В конце работы должны быть созданы файлы:

```text
data/prepared/final_sales_prepared.csv
data/output/final_practice_report.xlsx
data/output/figures/final_daily_revenue.png
data/output/figures/final_category_revenue.png
data/output/figures/final_top_regions.png
```

Также в конце ноутбука нужно написать **2–3 аналитических вывода**.

## Критерии зачета

| Критерий | Что проверяется |
|---|---|
| Загрузка | Загружены CSV, Excel, JSON, HTML |
| Качество | Проверены типы, пропуски, дубликаты, ключи |
| Очистка | Исправлены даты, числа, строки |
| Интеграция | Выполнены `merge` с товарами, регионами, клиентами и планом |
| Расчеты | Создан `revenue` и другие аналитические показатели |
| Анализ | Есть `groupby`, `agg`, `pivot_table` |
| Визуализация | Есть минимум 3 графика |
| Сохранение | CSV, Excel и PNG сохранены |
| Выводы | Есть 2–3 содержательных аналитических вывода |

# Часть 1. Подготовка окружения

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("pandas:", pd.__version__)

In [ ]:
def find_data_dir() -> Path:
    """Найти папку с исходными учебными данными."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "raw",
        current_dir.parent / "data" / "raw",
        current_dir.parent.parent / "data" / "raw",
    ]

    for candidate in candidates:
        if (candidate / "sales.csv").exists():
            return candidate

    return current_dir / "data" / "raw"


DATA_DIR = find_data_dir()
PREPARED_DIR = Path("data/prepared")
OUTPUT_DIR = Path("data/output")
FIGURES_DIR = OUTPUT_DIR / "figures"

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Папка с исходными данными:", DATA_DIR)
print("Папка для подготовленных данных:", PREPARED_DIR)
print("Папка для результатов:", OUTPUT_DIR)
print("Папка для графиков:", FIGURES_DIR)

## Проверка исходных файлов

In [ ]:
required_files = [
    "sales.csv",
    "products.xlsx",
    "regions.json",
    "clients.csv",
    "web_table_sample.html",
]

for file_name in required_files:
    file_path = DATA_DIR / file_name
    print(f"{file_name:<24} exists={file_path.exists()}")

# Часть 2. Загрузка данных

## Задание 1

Загрузите все источники:

| Переменная | Файл |
|---|---|
| `sales` | `sales.csv` |
| `products` | `products.xlsx`, лист `products` |
| `regions` | `regions.json` |
| `clients` | `clients.csv` |
| `plans` | первая таблица из `web_table_sample.html` |

In [ ]:
# TODO: загрузите данные

# sales = pd.read_csv(DATA_DIR / "sales.csv")
# products = pd.read_excel(DATA_DIR / "products.xlsx", sheet_name="products")
# regions = pd.read_json(DATA_DIR / "regions.json")
# clients = pd.read_csv(DATA_DIR / "clients.csv")
# plans = pd.read_html(DATA_DIR / "web_table_sample.html")[0]

## Задание 2. Проверьте размер таблиц

In [ ]:
# После загрузки раскомментируйте и выполните

# datasets = {
#     "sales": sales,
#     "products": products,
#     "regions": regions,
#     "clients": clients,
#     "plans": plans,
# }

# for name, df in datasets.items():
#     print(f"{name:<10} rows={df.shape[0]:>4}, columns={df.shape[1]:>3}")

## Задание 3. Посмотрите первые строки каждой таблицы

In [ ]:
# display(sales.head())
# display(products.head())
# display(regions.head())
# display(clients.head())
# display(plans.head())

# Часть 3. Проверка качества данных

## Задание 4

Для каждой таблицы проверьте:

- `head()`;
- `shape`;
- `columns`;
- `info()`;
- `isna().sum()`;
- `duplicated().sum()`.

In [ ]:
# Проверка sales

# sales.head()
# sales.shape
# sales.columns.tolist()
# sales.info()
# sales.isna().sum()
# sales.duplicated().sum()

In [ ]:
# Проверка products

# products.head()
# products.shape
# products.columns.tolist()
# products.info()
# products.isna().sum()
# products.duplicated().sum()

In [ ]:
# Проверка regions, clients, plans

# regions.info()
# clients.info()
# plans.info()

## Вопросы для самопроверки

Ответьте перед очисткой:

1. Какие столбцы должны быть числовыми?
2. Какие столбцы должны быть датами?
3. Где есть пропуски?
4. Есть ли дубликаты по ключам `sale_id`, `product_id`, `client_id`, `region_id`?
5. Какие строки или значения выглядят подозрительно?

# Часть 4. Минимальная очистка данных

## Задание 5. Создайте рабочие копии

In [ ]:
# sales_work = sales.copy()
# products_work = products.copy()
# regions_work = regions.copy()
# clients_work = clients.copy()
# plans_work = plans.copy()

## Вспомогательная функция для дат

In [ ]:
def parse_dates_safely(series: pd.Series) -> pd.Series:
    """Преобразовать даты с учетом разных версий pandas."""
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)
    except TypeError:
        return pd.to_datetime(series, errors="coerce", dayfirst=True)

## Задание 6. Очистите продажи

Нужно:

1. Преобразовать `order_date` к дате.
2. Создать `month` в формате `YYYY-MM`.
3. Очистить `channel`.
4. Преобразовать `quantity`, `unit_price`, `discount_percent` к числам.
5. Заполнить пропуски в `discount_percent` нулем.
6. Удалить дубликаты по `sale_id`.

In [ ]:
# sales_work["order_date"] = parse_dates_safely(sales_work["order_date"])
# sales_work["month"] = sales_work["order_date"].dt.to_period("M").astype(str)

# sales_work["channel"] = (
#     sales_work["channel"]
#     .astype("string")
#     .str.strip()
#     .str.lower()
# )

# sales_work["quantity"] = pd.to_numeric(sales_work["quantity"], errors="coerce")
# sales_work["unit_price"] = pd.to_numeric(sales_work["unit_price"], errors="coerce")
# sales_work["discount_percent"] = pd.to_numeric(sales_work["discount_percent"], errors="coerce").fillna(0)

# sales_work = sales_work.drop_duplicates(subset=["sale_id"], keep="first")

# sales_work.info()
# sales_work.head()

## Задание 7. Очистите товары

Нужно:

1. Очистить `category`.
2. Преобразовать `purchase_price` к числу.
3. Удалить дубликаты по `product_id`.

In [ ]:
# products_work["category"] = (
#     products_work["category"]
#     .astype("string")
#     .str.strip()
#     .str.lower()
# )

# products_work["purchase_price"] = pd.to_numeric(products_work["purchase_price"], errors="coerce")
# products_work = products_work.drop_duplicates(subset=["product_id"], keep="first")

# print("Дубликаты product_id:", products_work.duplicated(subset=["product_id"]).sum())
# products_work.head()

## Задание 8. Очистите регионы, клиентов и план

Для `regions_work`:

- очистить `federal_district`.

Для `clients_work`:

- очистить `client_type`;
- преобразовать `registration_date`;
- удалить дубликаты по `client_id`.

Для `plans_work`:

- очистить `channel`;
- преобразовать `sales_plan`, `orders_plan`;
- привести `month` к формату `YYYY-MM`.

In [ ]:
# regions_work["federal_district"] = (
#     regions_work["federal_district"]
#     .astype("string")
#     .str.strip()
#     .str.lower()
# )

# clients_work["client_type"] = (
#     clients_work["client_type"]
#     .astype("string")
#     .str.strip()
#     .str.upper()
# )
# clients_work["registration_date"] = parse_dates_safely(clients_work["registration_date"])
# clients_work = clients_work.drop_duplicates(subset=["client_id"], keep="first")

# plans_work["channel"] = (
#     plans_work["channel"]
#     .astype("string")
#     .str.strip()
#     .str.lower()
# )
# plans_work["sales_plan"] = pd.to_numeric(plans_work["sales_plan"], errors="coerce")
# plans_work["orders_plan"] = pd.to_numeric(plans_work["orders_plan"], errors="coerce")
# plans_work["month_dt"] = parse_dates_safely(plans_work["month"].astype("string"))
# plans_work["month"] = plans_work["month_dt"].dt.to_period("M").astype(str)
# plans_work = plans_work.drop(columns=["month_dt"])

# plans_work.head()

# Часть 5. Проверка ключей перед объединением

## Задание 9

Проверьте, какие ключи из продаж отсутствуют в справочниках:

- `product_id`;
- `client_id`;
- `region_id`.

In [ ]:
# missing_product_ids = sorted(
#     set(sales_work["product_id"].dropna().unique()) -
#     set(products_work["product_id"].dropna().unique())
# )

# missing_client_ids = sorted(
#     set(sales_work["client_id"].dropna().unique()) -
#     set(clients_work["client_id"].dropna().unique())
# )

# missing_region_ids = sorted(
#     set(sales_work["region_id"].dropna().unique()) -
#     set(regions_work["region_id"].dropna().unique())
# )

# print("missing_product_ids:", missing_product_ids)
# print("missing_client_ids:", missing_client_ids)
# print("missing_region_ids:", missing_region_ids)

## Вопрос

Что произойдет после `left merge`, если ключ есть в продажах, но отсутствует в справочнике?

**Ваш ответ:**  
...

# Часть 6. Объединение данных

## Задание 10. Продажи + товары

Объедините `sales_work` и `products_work` по `product_id`.

Используйте:

```python
how="left"
indicator=True
validate="many_to_one"
```

Проверьте количество строк до и после.

In [ ]:
# rows_before = sales_work.shape[0]

# sales_products = sales_work.merge(
#     products_work,
#     on="product_id",
#     how="left",
#     indicator=True,
#     validate="many_to_one"
# )

# rows_after = sales_products.shape[0]

# print("rows_before:", rows_before)
# print("rows_after:", rows_after)
# print(sales_products["_merge"].value_counts())

# sales_products[sales_products["_merge"] == "left_only"][
#     ["sale_id", "product_id", "product_name", "category"]
# ]

In [ ]:
# После проверки удалите _merge

# sales_products = sales_products.drop(columns=["_merge"])

## Задание 11. Добавьте регионы

In [ ]:
# rows_before = sales_products.shape[0]

# sales_products_regions = sales_products.merge(
#     regions_work,
#     on="region_id",
#     how="left",
#     indicator=True,
#     validate="many_to_one"
# )

# rows_after = sales_products_regions.shape[0]

# print("rows_before:", rows_before)
# print("rows_after:", rows_after)
# print(sales_products_regions["_merge"].value_counts())

# sales_products_regions = sales_products_regions.drop(columns=["_merge"])

## Задание 12. Добавьте клиентов

In [ ]:
# rows_before = sales_products_regions.shape[0]

# sales_full = sales_products_regions.merge(
#     clients_work,
#     on="client_id",
#     how="left",
#     indicator=True,
#     validate="many_to_one"
# )

# rows_after = sales_full.shape[0]

# print("rows_before:", rows_before)
# print("rows_after:", rows_after)
# print(sales_full["_merge"].value_counts())

# sales_full = sales_full.drop(columns=["_merge"])

## Задание 13. Добавьте план продаж

План объединяется по трем ключам:

```text
region_id + channel + month
```

In [ ]:
# merge_keys = ["region_id", "channel", "month"]

# sales_with_plan = sales_full.merge(
#     plans_work,
#     on=merge_keys,
#     how="left",
#     indicator=True,
#     validate="many_to_one"
# )

# print(sales_with_plan["_merge"].value_counts())

# sales_with_plan[sales_with_plan["_merge"] == "left_only"][
#     ["sale_id", "region_id", "region_name", "channel", "month", "sales_plan", "orders_plan"]
# ].head(20)

# sales_with_plan = sales_with_plan.drop(columns=["_merge"])

# Часть 7. Создание revenue и расчетных показателей

## Задание 14. Создайте итоговую таблицу

In [ ]:
# final_df = sales_with_plan.copy()

## Задание 15. Создайте расчетные показатели

Создайте:

```text
gross_revenue = quantity * unit_price
discount_amount = gross_revenue * discount_percent / 100
revenue = gross_revenue - discount_amount
purchase_cost = quantity * purchase_price
gross_profit = revenue - purchase_cost
```

In [ ]:
# final_df["gross_revenue"] = final_df["quantity"] * final_df["unit_price"]
# final_df["discount_amount"] = final_df["gross_revenue"] * final_df["discount_percent"] / 100
# final_df["revenue"] = final_df["gross_revenue"] - final_df["discount_amount"]
# final_df["purchase_cost"] = final_df["quantity"] * final_df["purchase_price"]
# final_df["gross_profit"] = final_df["revenue"] - final_df["purchase_cost"]

# final_df[[
#     "sale_id", "quantity", "unit_price", "discount_percent",
#     "gross_revenue", "discount_amount", "revenue",
#     "purchase_cost", "gross_profit"
# ]].head()

## Задание 16. Проверьте итоговую таблицу

In [ ]:
# print("shape:", final_df.shape)
# print("duplicates sale_id:", final_df.duplicated(subset=["sale_id"]).sum())

# quality_checks = {
#     "revenue <= 0": (final_df["revenue"] <= 0).sum(),
#     "missing product_name": final_df["product_name"].isna().sum(),
#     "missing region_name": final_df["region_name"].isna().sum(),
#     "missing client_type": final_df["client_type"].isna().sum(),
#     "missing sales_plan": final_df["sales_plan"].isna().sum(),
# }

# for check, value in quality_checks.items():
#     print(f"{check:<24}: {value}")

# Часть 8. Аналитические показатели

## Задание 17. Показатели по категориям

Посчитайте `category_summary`:

- количество заказов;
- количество товаров;
- выручка;
- прибыль;
- средняя скидка;
- средний чек.

In [ ]:
# category_summary = (
#     final_df
#     .groupby("category", dropna=False)
#     .agg(
#         orders_count=("sale_id", "nunique"),
#         total_quantity=("quantity", "sum"),
#         total_revenue=("revenue", "sum"),
#         total_profit=("gross_profit", "sum"),
#         avg_discount=("discount_percent", "mean"),
#     )
#     .reset_index()
# )

# category_summary["avg_check"] = category_summary["total_revenue"] / category_summary["orders_count"]

# category_summary = category_summary.sort_values("total_revenue", ascending=False)

# category_summary

## Задание 18. Показатели по регионам и каналам

Создайте `region_channel_summary`.

In [ ]:
# region_channel_summary = (
#     final_df
#     .groupby(["region_name", "channel"], dropna=False)
#     .agg(
#         orders_count=("sale_id", "nunique"),
#         total_revenue=("revenue", "sum"),
#         total_profit=("gross_profit", "sum"),
#     )
#     .reset_index()
# )

# region_channel_summary["avg_check"] = (
#     region_channel_summary["total_revenue"] / region_channel_summary["orders_count"]
# )

# region_channel_summary = region_channel_summary.sort_values("total_revenue", ascending=False)

# region_channel_summary.head(20)

## Задание 19. Сводная таблица

Создайте `revenue_pivot`:

- строки — `region_name`;
- столбцы — `category`;
- значения — `revenue`;
- функция — сумма;
- пустые значения — 0.

In [ ]:
# revenue_pivot = pd.pivot_table(
#     final_df,
#     values="revenue",
#     index="region_name",
#     columns="category",
#     aggfunc="sum",
#     fill_value=0,
# )

# revenue_pivot

## Задание 20. Дополнительно для сильных

Посчитайте выполнение плана по регионам и каналам.

Подумайте, как не сложить один и тот же план много раз.

In [ ]:
# plan_summary = (
#     final_df
#     .groupby(["region_name", "channel", "month"], dropna=False)
#     .agg(
#         total_revenue=("revenue", "sum"),
#         sales_plan=("sales_plan", "max"),
#         orders_count=("sale_id", "nunique"),
#     )
#     .reset_index()
# )

# plan_summary["plan_completion_rate"] = plan_summary["total_revenue"] / plan_summary["sales_plan"]

# plan_summary.sort_values("plan_completion_rate", ascending=False).head(20)

# Часть 9. Визуализация

## Подготовьте таблицу для графиков

In [ ]:
# viz_df = final_df[
#     final_df["order_date"].notna() &
#     final_df["revenue"].notna() &
#     (final_df["revenue"] > 0)
# ].copy()

# viz_df.shape

## Задание 21. График динамики выручки

In [ ]:
# daily_revenue = (
#     viz_df
#     .groupby("order_date", as_index=False)
#     .agg(total_revenue=("revenue", "sum"))
#     .sort_values("order_date")
# )

# plt.figure(figsize=(10, 5))
# plt.plot(daily_revenue["order_date"], daily_revenue["total_revenue"], marker="o")
# plt.title("Динамика выручки по датам")
# plt.xlabel("Дата заказа")
# plt.ylabel("Выручка")
# plt.xticks(rotation=45)
# plt.grid(True)
# plt.tight_layout()

# daily_revenue_path = FIGURES_DIR / "final_daily_revenue.png"
# plt.savefig(daily_revenue_path, dpi=150)
# plt.show()

# print("Сохранено:", daily_revenue_path)

## Задание 22. График выручки по категориям

In [ ]:
# plt.figure(figsize=(10, 5))
# plt.bar(category_summary["category"].astype(str), category_summary["total_revenue"])
# plt.title("Выручка по категориям")
# plt.xlabel("Категория")
# plt.ylabel("Выручка")
# plt.xticks(rotation=45)
# plt.tight_layout()

# category_revenue_path = FIGURES_DIR / "final_category_revenue.png"
# plt.savefig(category_revenue_path, dpi=150)
# plt.show()

# print("Сохранено:", category_revenue_path)

## Задание 23. График топ-10 регионов по выручке

In [ ]:
# top_regions = (
#     viz_df
#     .groupby("region_name", dropna=False, as_index=False)
#     .agg(total_revenue=("revenue", "sum"))
#     .sort_values("total_revenue", ascending=False)
#     .head(10)
# )

# top_regions_for_plot = top_regions.sort_values("total_revenue")

# plt.figure(figsize=(10, 6))
# plt.barh(top_regions_for_plot["region_name"].astype(str), top_regions_for_plot["total_revenue"])
# plt.title("Топ регионов по выручке")
# plt.xlabel("Выручка")
# plt.ylabel("Регион")
# plt.tight_layout()

# top_regions_path = FIGURES_DIR / "final_top_regions.png"
# plt.savefig(top_regions_path, dpi=150)
# plt.show()

# print("Сохранено:", top_regions_path)

## Задание 24. Дополнительные графики

In [ ]:
# Гистограмма чеков

# plt.figure(figsize=(10, 5))
# plt.hist(viz_df["revenue"].dropna(), bins=15)
# plt.title("Распределение суммы заказа")
# plt.xlabel("Сумма заказа")
# plt.ylabel("Количество заказов")
# plt.tight_layout()
# plt.show()

In [ ]:
# Scatter plot: revenue и gross_profit

# plt.figure(figsize=(8, 5))
# plt.scatter(viz_df["revenue"], viz_df["gross_profit"])
# plt.title("Связь выручки и валовой прибыли")
# plt.xlabel("Выручка")
# plt.ylabel("Валовая прибыль")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# Часть 10. Сохранение результатов

## Задание 25. Сохраните итоговую таблицу

In [ ]:
# final_prepared_path = PREPARED_DIR / "final_sales_prepared.csv"

# final_df.to_csv(final_prepared_path, index=False, encoding="utf-8")

# print("Сохранено:", final_prepared_path)
# print("Файл существует:", final_prepared_path.exists())

## Задание 26. Сохраните Excel-отчет

In [ ]:
# final_report_path = OUTPUT_DIR / "final_practice_report.xlsx"

# with pd.ExcelWriter(final_report_path, engine="openpyxl") as writer:
#     final_df.head(50).to_excel(writer, sheet_name="final_sales_sample", index=False)
#     category_summary.to_excel(writer, sheet_name="category_summary", index=False)
#     region_channel_summary.to_excel(writer, sheet_name="region_channel", index=False)
#     revenue_pivot.to_excel(writer, sheet_name="revenue_pivot")

#     # Если сделали plan_summary, раскомментируйте:
#     # plan_summary.to_excel(writer, sheet_name="plan_summary", index=False)

# print("Сохранено:", final_report_path)
# print("Файл существует:", final_report_path.exists())

# Часть 11. Самопроверка

In [ ]:
objects_to_check = [
    "sales", "products", "regions", "clients", "plans",
    "sales_work", "products_work", "regions_work", "clients_work", "plans_work",
    "final_df", "category_summary", "region_channel_summary", "revenue_pivot",
]

for object_name in objects_to_check:
    print(f"{object_name:<24}", "OK" if object_name in globals() else "НЕ СОЗДАНО")

In [ ]:
files_to_check = [
    PREPARED_DIR / "final_sales_prepared.csv",
    OUTPUT_DIR / "final_practice_report.xlsx",
    FIGURES_DIR / "final_daily_revenue.png",
    FIGURES_DIR / "final_category_revenue.png",
    FIGURES_DIR / "final_top_regions.png",
]

for file_path in files_to_check:
    print(f"{file_path} exists={file_path.exists()}")

## Финальный чек-лист

- [ ] Все 5 файлов загружены.
- [ ] Проверены типы, пропуски и дубликаты.
- [ ] Даты, числа и строки очищены.
- [ ] Проверены ключи перед объединением.
- [ ] Выполнены все `merge`.
- [ ] Проверены строки до и после объединений.
- [ ] Создан `revenue`.
- [ ] Созданы `category_summary`, `region_channel_summary`, `revenue_pivot`.
- [ ] Построены минимум 3 графика.
- [ ] Итоговый CSV сохранен.
- [ ] Excel-отчет сохранен.
- [ ] Написаны 2–3 аналитических вывода.

# Часть 12. Аналитические выводы

Напишите 2–3 вывода. Вывод должен быть связан с расчетами или графиками.

Плохо:

> Я построил график.

Хорошо:

> Категория X лидирует по выручке, но по прибыли уступает категории Y. Это значит, что при управленческом анализе нужно смотреть не только оборот, но и прибыльность.

---

## Вывод 1

...

## Вывод 2

...

## Вывод 3

...

# Что сдавать преподавателю

1. Заполненный ноутбук `07_final_practice_case.ipynb`.
2. `data/prepared/final_sales_prepared.csv`.
3. `data/output/final_practice_report.xlsx`.
4. Графики:
   - `final_daily_revenue.png`;
   - `final_category_revenue.png`;
   - `final_top_regions.png`.
5. 2–3 аналитических вывода в конце ноутбука.